In [1]:
# SKlearn Pipelines (Pandas)

In [2]:
# The difference between sklearn pipelines and transformers is 
# that a pipeline is a sequence of steps. A transformer transforms
# the data, and a pipeline is a sequence of transformers.
# A ColumnTransformer applies multiple transformers to different
# columns of the input data.
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer
from sklearn.base import BaseEstimator, TransformerMixin
# Custom imports
from get_dataset_pandas import get_pandas_df
from process_data_pandas import tweak_housing

In [3]:
# Gets Pandas dataframe
raw = get_pandas_df()

In [4]:
# See what the numeric columns are.
print(list(raw.select_dtypes(include=[np.number]).columns.values))

['id', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long', 'sqft_living15', 'sqft_lot15', 'date_year', 'date_month', 'date_day']


In [5]:
# Standardises the scales
"""
Imagine comparing number of bedrooms or bathrooms to square feet.
If these are not standardised, some algorithms may pay more attention to square feet
since the numbers are much larger than the no. of rooms.

StandardScaler's fit calculates the mean and standard deviation for each numeric column.
These values are stored internally in the scaler object, std.mean_, std.scale_ etc.

transform produces standardised data with mean that approximates to 0 and varience that 
approximates to 1.

The following codes has additional print statements to work with Python's IDLE.
"""
# Define numeric features
numeric_features = ['bedrooms', 'bathrooms', 'sqft_living']
# Apply StandardScaler
std = StandardScaler()
std_f = std.fit_transform(raw[numeric_features])
print(std_f)

[[-0.39873715 -1.44746357 -0.97983502]
 [-0.39873715  0.1756067   0.53363434]
 [-1.47395936 -1.44746357 -1.42625404]
 ...
 [-1.47395936 -1.77207762 -1.15404732]
 [-0.39873715  0.50022075 -0.52252773]
 [-1.47395936 -1.77207762 -1.15404732]]


In [6]:
# Build pipeline
num_pipeline = Pipeline([
    ('std', StandardScaler())
])

# Fit and transform
num_pipe = num_pipeline.fit_transform(
    raw[numeric_features]
)
print(num_pipe)

[[-0.39873715 -1.44746357 -0.97983502]
 [-0.39873715  0.1756067   0.53363434]
 [-1.47395936 -1.44746357 -1.42625404]
 ...
 [-1.47395936 -1.77207762 -1.15404732]
 [-0.39873715  0.50022075 -0.52252773]
 [-1.47395936 -1.77207762 -1.15404732]]


In [7]:
# Add another step
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')), # if a value is missing, add the median value
    ('std', StandardScaler())])

# Fit and transform
num_pipe = num_pipeline.fit_transform(
    tweak_housing(raw)[numeric_features]
)
print(num_pipe)

[[-0.39873715 -1.44746357 -0.97983502]
 [-0.39873715  0.1756067   0.53363434]
 [-1.47395936 -1.44746357 -1.42625404]
 ...
 [-1.47395936 -1.77207762 -1.15404732]
 [-0.39873715  0.50022075 -0.52252773]
 [-1.47395936 -1.77207762 -1.15404732]]


In [8]:
"""
i.e., if we train it on a dataset that has certain categories and when we try
to do prediction, if it comes across a new category, we ignore it

max_categories sets maximum no. of columns
"""
cat_features = ['zipcode']

ohe = OneHotEncoder(handle_unknown='ignore',
                    sparse_output=False, max_categories=10)

# Fit and transform
ohe_t = ohe.fit_transform(
    tweak_housing(raw)[cat_features]
    )
print(ohe_t)

[[0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 1.]
 ...
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 1.]]


In [9]:
# Transformer from a function.
tweak_transformer = FunctionTransformer(tweak_housing)
print(tweak_transformer.fit_transform(raw))

               id     price  bedrooms  bathrooms  sqft_living  sqft_lot  \
0      7129300520  221900.0         3       1.00         1180      5650   
1      6414100192  538000.0         3       2.25         2570      7242   
2      5631500400  180000.0         2       1.00          770     10000   
3      2487200875  604000.0         4       3.00         1960      5000   
4      1954400510  510000.0         3       2.00         1680      8080   
...           ...       ...       ...        ...          ...       ...   
21608   263000018  360000.0         3       2.50         1530      1131   
21609  6600060120  400000.0         4       2.50         2310      5813   
21610  1523300141  402101.0         2       0.75         1020      1350   
21611   291310100  400000.0         3       2.50         1600      2388   
21612  1523300157  325000.0         2       0.75         1020      1076   

       floors  waterfront  view  condition  ...  sqft_above  sqft_basement  \
0         1.0        

In [10]:
categorical_features = ['zipcode']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

# Column Transformer lets us apply specific transformations to certain columns.
ct = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore',
                              sparse_output=False), categorical_features)])

ct_f = ct.fit_transform(
    tweak_housing(raw)[[*numeric_features, *cat_features]]
)
print(ct_f)

[[-0.39873715 -1.44746357 -0.97983502 ...  0.          0.
   0.        ]
 [-0.39873715  0.1756067   0.53363434 ...  0.          0.
   0.        ]
 [-1.47395936 -1.44746357 -1.42625404 ...  0.          0.
   0.        ]
 ...
 [-1.47395936 -1.77207762 -1.15404732 ...  0.          0.
   0.        ]
 [-0.39873715  0.50022075 -0.52252773 ...  0.          0.
   0.        ]
 [-1.47395936 -1.77207762 -1.15404732 ...  0.          0.
   0.        ]]


In [11]:
# Custom transformer that maps a zip code to the average price of that zip code.
class ZipAvgPriceAdder(BaseEstimator, TransformerMixin):
    
    def __init__(self):
            pass

    # Assume X is a pandas dataframe.
    # Group X by the zip code, then aggregate that to get the average price.   
    def fit(self, X, y=None):
        self.zip_avg_price = X.groupby('zipcode')['price'].mean().reset_index()
        return self
    
    # Transform data, where X is the dataframe.
    # Here we are going to add the zip price average information on it.
    def transform(self, X, y=None):
        return (X.merge(self.zip_avg_price, on='zipcode', suffixes=('', '_zip_mean'))
                .rename(columns={'price_zip_mean':'zip_mean'}))

zip_adder = ZipAvgPriceAdder()
zip_f = zip_adder.fit_transform(raw[['zipcode', 'price']])
print(zip_f)

      zipcode     price       zip_mean
0       98178  221900.0  310612.755725
1       98125  538000.0  469455.770732
2       98028  180000.0  462480.035336
3       98136  604000.0  551688.673004
4       98074  510000.0  685605.775510
...       ...       ...            ...
21608   98103  360000.0  584919.210963
21609   98146  400000.0  359483.239583
21610   98144  402101.0  594547.650146
21611   98027  400000.0  616990.592233
21612   98144  325000.0  594547.650146

[21613 rows x 3 columns]


### Full example continued at 1d_ii_pipelines.pandas.ipynb.